# HEP Data

This noteboook reads in the outputs of combine to prepare tables of limits of the different points in the parameter space.

In [1]:
import numpy as np
import glob
import os
import uproot
import json
from tabulate import tabulate
import plot_utils as utils
import json
import matplotlib.tri as tri

In [2]:
def create_limit_table(params):
    """
    Expects params to be the output for get_scan_limits.
    Returns table of limits.
    """
    rows = np.array(['mS', 'mPhi', 'T', 'mA', 'Xsec', 'Obs', 'Exp', '+2sigma', '+1sigma', '-1sigma', '-2sigma'])
    for (param, lims) in params:

        _exp =lims[1][2]
        _s1p =lims[1][1] 
        _s1m =lims[1][3] 
        _s2p =lims[1][0] 
        _s2m =lims[1][4] 
        _obs =lims[1][5] 

        mS, mPhi, T, decay, xsec = param

        if decay.lower() == 'leptonic': mA = 0.7
        elif decay.lower() == 'hadronic': mA = 0.5
        elif decay.lower() == 'generic': mA = 1.0
        else: raise ValueError(f"Unrecognized decay mode: {decay}")  

        newline = [mS, mPhi, T, mA, xsec, _obs, _exp, _s2p, _s1p, _s1m, _s2m]
        rows = np.row_stack((rows, newline))
    
    return rows

In [2]:
# script parameters
ofile = '../approval_higherPrecision/hep_{}_{}.txt'
itag = '../approval_higherPrecision/'
HT = '1000'

In [4]:
# asymptotic_params = plot.get_scan_limits(path=itag, file='../config/xsections_SUEP.json', method='AsymptoticLimits')
# asymptotic_table = create_limit_table(asymptotic_params)

In [11]:
toys_params = plot.get_scan_limits(path=itag, file='../config/xsections_SUEP.json', method='HybridNew')
toys_table = create_limit_table(toys_params)

In [7]:
# decays = ['generic', 'leptonic', 'hadronic']

# # divide the table by decay mode, and save each separetely
# for decay in decays:
    
#     good_rows = asymptotic_table[:,3]==decay  # only select one decay mode
#     good_rows[0] = True          # also save the labels
#     table_decay = asymptotic_table[good_rows, :] 
#     table_decay = table_decay[:, np.arange(asymptotic_table.shape[1]) != 3] # exclude the decay column from output table

#     fname = ofile.format('AsymptoticLimits', decay)
#     with open(fname, "w") as file:
#         file.write(tabulate(table_decay[1:], headers=table_decay[0,:]))

#    print("Written", fname)

Written ../approval_higherPrecision/hep_AsymptoticLimits_generic.txt
Written ../approval_higherPrecision/hep_AsymptoticLimits_leptonic.txt
Written ../approval_higherPrecision/hep_AsymptoticLimits_hadronic.txt


In [15]:
decays = ['0.5', '0.7', '1.0']

# divide the table by decay mode, and save each separetely
for decay in decays:
    
    good_rows = toys_table[:,3]==decay  # only select one decay mode
    good_rows[0] = True          # also save the labels
    table_decay = toys_table[good_rows, :] 
    table_decay = table_decay[:, np.arange(toys_table.shape[1]) != 3] # exclude the decay column from output table

    fname = ofile.format('HybridNew', 'mAp'+decay)
    with open(fname, "w") as file:
        file.write(tabulate(table_decay[1:], headers=table_decay[0,:]))

    print("Written", fname)

# save the full table
with open(ofile.format('HybridNew', 'full'), "w") as file:
    file.write(tabulate(toys_table[1:], headers=toys_table[0,:]))
    print("Written", ofile.format('HybridNew', 'full'))

Written ../approval_higherPrecision/hep_HybridNew_mAp0.5.txt
Written ../approval_higherPrecision/hep_HybridNew_mAp0.7.txt
Written ../approval_higherPrecision/hep_HybridNew_mAp1.0.txt
Written ../approval_higherPrecision/hep_HybridNew_full.txt


In [7]:
fig4 = utils.plot_summary_limits_mPhi_temp(decay='generic', path='../approval_higherPrecision/', method='HybridNew', returnData=True)
with open(os.path.join(itag, 'figure_4.json'), 'w') as f:
    f.write(json.dumps(fig4))

sup_fig3 = utils.plot_summary_limits_mPhi_temp(decay='leptonic', path='../approval_higherPrecision/', method='HybridNew', returnData=True)
with open(os.path.join(itag, 'sup_figure_3.json'), 'w') as f:
    f.write(json.dumps(sup_fig3))

sup_fig4 = utils.plot_summary_limits_mPhi_temp(decay='hadronic', path='../approval_higherPrecision/', method='HybridNew', returnData=True)
with open(os.path.join(itag, 'sup_figure_4.json'), 'w') as f:
    f.write(json.dumps(sup_fig4))

In [4]:
sup_fig2 = {
    'mS300': {},
    'mS600': {},
    'mS1000': {},
}

triang, obs = utils.plot_mPhi_temp_limits(ms=300, decay='generic', tricontour='log', path=itag, method='HybridNew', returnContour=True)
grid_x = np.linspace(2, 8, 10)
grid_y = np.linspace(0, 6, 10)
grid_X, grid_Y = np.meshgrid(grid_x, grid_y)
grid_Z = tri.LinearTriInterpolator(triang, obs)(grid_X, grid_Y)
extracted_x = grid_X.flatten()
extracted_y = grid_Y.flatten()
extracted_z = 10**grid_Z.flatten()
sup_fig2['mS300']['mPhi'] = extracted_x.tolist()
sup_fig2['mS300']['T'] = extracted_y.tolist()
sup_fig2['mS300']['xsec'] = extracted_z.tolist()

fig = utils.plot_mPhi_temp_limits(ms=600, decay='generic', tricontour='log', path=itag, showPoints=False, method='HybridNew', showTheoryLines=True)
grid_x = np.linspace(2, 8, 10)
grid_y = np.linspace(0, 11, 10)
grid_X, grid_Y = np.meshgrid(grid_x, grid_y)
grid_Z = tri.LinearTriInterpolator(triang, obs)(grid_X, grid_Y)
extracted_x = grid_X.flatten()
extracted_y = grid_Y.flatten()
extracted_z = 10**grid_Z.flatten()
sup_fig2['mS600']['mPhi'] = extracted_x.tolist()
sup_fig2['mS600']['T'] = extracted_y.tolist()
sup_fig2['mS600']['xsec'] = extracted_z.tolist()

fig = utils.plot_mPhi_temp_limits(ms=1000, decay='generic', tricontour='log', path=itag, showPoints=False, method='HybridNew', showTheoryLines=True)
grid_x = np.linspace(2, 8, 10)
grid_y = np.linspace(0, 15, 10)
grid_X, grid_Y = np.meshgrid(grid_x, grid_y)
grid_Z = tri.LinearTriInterpolator(triang, obs)(grid_X, grid_Y)
extracted_x = grid_X.flatten()
extracted_y = grid_Y.flatten()
extracted_z = 10**grid_Z.flatten()
sup_fig2['mS1000']['mPhi'] = extracted_x.tolist()
sup_fig2['mS1000']['T'] = extracted_y.tolist()
sup_fig2['mS1000']['xsec'] = extracted_z.tolist()

with open(os.path.join(itag, 'sup_figure_2.json'), 'w') as f:
    f.write(json.dumps(sup_fig2))